# Step 2  Sakila Data Warehouse — ETL Process
**Source (OLTP):** `sakila` MySQL database  
**Destination (OLAP):** `sakila_dw` MySQL database  

**Flow:**
```
sakila (OLTP)
   ↓  EXTRACT   — read tables with pandas + SQLAlchemy
   ↓  TRANSFORM — clean, join, deduplicate, compute measures
   ↓  LOAD      — write to sakila_dw star schema
```
**Run:** Kernel → Restart & Run All

---
## 0 — Install Required Libraries

In [1]:
import sys
!{sys.executable} -m pip install pandas sqlalchemy pymysql numpy --quiet --trusted-host pypi.org --trusted-host files.pythonhosted.org

---
## 1 — Imports & Database Connections

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# ── connection settings ──────────────────────────────────────
HOST     = '127.0.0.1'
PORT     = 3306
USER     = 'root'
PASSWORD = 'ramasabitbel'  

# source OLTP database
oltp_engine = create_engine(
    f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/sakila',
    echo=False
)

# destination OLAP data warehouse
dw_engine = create_engine(
    f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/sakila_dw',
    echo=False
)

# test both connections
with oltp_engine.connect() as c:
    print('OLTP connected:', c.execute(text('SELECT DATABASE()')).scalar())
with dw_engine.connect() as c:
    print('DW   connected:', c.execute(text('SELECT DATABASE()')).scalar())

OLTP connected: sakila
DW   connected: sakila_dw


---
## 2 — EXTRACT
Read every relevant OLTP table into a pandas DataFrame.

In [3]:
customer  = pd.read_sql('SELECT * FROM customer',      oltp_engine)
address   = pd.read_sql('SELECT * FROM address',       oltp_engine)
city      = pd.read_sql('SELECT * FROM city',          oltp_engine)
country   = pd.read_sql('SELECT * FROM country',       oltp_engine)
film      = pd.read_sql('SELECT * FROM film',          oltp_engine)
language  = pd.read_sql('SELECT * FROM language',      oltp_engine)
category  = pd.read_sql('SELECT * FROM category',      oltp_engine)
film_cat  = pd.read_sql('SELECT * FROM film_category', oltp_engine)
store     = pd.read_sql('SELECT * FROM store',         oltp_engine)
staff     = pd.read_sql('SELECT * FROM staff',         oltp_engine)
actor     = pd.read_sql('SELECT * FROM actor',         oltp_engine)
film_act  = pd.read_sql('SELECT * FROM film_actor',    oltp_engine)
inventory = pd.read_sql('SELECT * FROM inventory',     oltp_engine)
rental    = pd.read_sql('SELECT * FROM rental',        oltp_engine)
payment   = pd.read_sql('SELECT * FROM payment',       oltp_engine)

print('Extract complete:')
for name, df in [('customer',customer),('address',address),('city',city),
                 ('country',country),('film',film),('language',language),
                 ('category',category),('film_cat',film_cat),('store',store),
                 ('staff',staff),('actor',actor),('film_act',film_act),
                 ('inventory',inventory),('rental',rental),('payment',payment)]:
    print(f'  {name:<12}: {len(df):>6} rows')

Extract complete:
  customer    :    599 rows
  address     :    603 rows
  city        :    600 rows
  country     :    109 rows
  film        :   1000 rows
  language    :      6 rows
  category    :     16 rows
  film_cat    :   1000 rows
  store       :      2 rows
  staff       :      2 rows
  actor       :    200 rows
  film_act    :   5462 rows
  inventory   :   4581 rows
  rental      :  16044 rows
  payment     :  16044 rows


---
## 3 — TRANSFORM
Clean, join, deduplicate, and compute new columns.

### 3.1 — dim_date
Generated from a date range — not pulled from OLTP.

In [4]:
date_range = pd.date_range(start='2005-01-01', end='2010-12-31', freq='D')
dim_date   = pd.DataFrame({'full_date': date_range})

dim_date['date_key']     = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['day_of_week']  = dim_date['full_date'].dt.weekday + 1
dim_date['day_name']     = dim_date['full_date'].dt.day_name()
dim_date['day_of_month'] = dim_date['full_date'].dt.day
dim_date['day_of_year']  = dim_date['full_date'].dt.day_of_year
dim_date['week_of_year'] = dim_date['full_date'].dt.isocalendar().week.astype(int)
dim_date['month_number'] = dim_date['full_date'].dt.month
dim_date['month_name']   = dim_date['full_date'].dt.month_name()
dim_date['quarter']      = dim_date['full_date'].dt.quarter
dim_date['year']         = dim_date['full_date'].dt.year
dim_date['is_weekend']   = (dim_date['full_date'].dt.weekday >= 5).astype(int)
dim_date = dim_date.drop_duplicates(subset='date_key')

print(f'dim_date rows: {len(dim_date)}')
dim_date.head(3)

dim_date rows: 2191


,full_date,date_key,day_of_week,day_name,day_of_month,day_of_year,week_of_year,month_number,month_name,quarter,year,is_weekend
0,2005-01-01,20050101,6,Saturday,1,1,53,1,January,1,2005,1
1,2005-01-02,20050102,7,Sunday,2,2,53,1,January,1,2005,1
2,2005-01-03,20050103,1,Monday,3,3,1,1,January,1,2005,0


### 3.2 — dim_customer
Source: customer + address + city + country (flattened).



In [5]:
# select only needed columns BEFORE merging to avoid last_update duplicate conflicts
customer_c = customer[['customer_id','first_name','last_name','email','address_id','active','create_date']]
address_c  = address [['address_id','address','district','postal_code','phone','city_id']]
city_c     = city    [['city_id','city','country_id']]
country_c  = country [['country_id','country']]

dim_customer = (
    customer_c
    .merge(address_c, on='address_id')
    .merge(city_c,    on='city_id')
    .merge(country_c, on='country_id')
)

# fill missing values
dim_customer['email']       = dim_customer['email'].fillna('no-email@unknown.com')
dim_customer['postal_code'] = dim_customer['postal_code'].fillna('00000')
dim_customer['active']      = dim_customer['active'].astype(int)
dim_customer = dim_customer.drop_duplicates(subset='customer_id')


dim_customer['create_date'] = pd.to_datetime(dim_customer['create_date']).dt.date

# SCD Type 2 columns
dim_customer['row_effective_date'] = pd.Timestamp.today().normalize()
dim_customer['row_expiry_date']    = None
dim_customer['is_current']         = 1

# keep only columns that exist in the DW table (drop join key residuals)
dim_customer = dim_customer[[
    'customer_id','first_name','last_name','email',
    'address','district','city','country',
    'postal_code','phone','active','create_date',
    'row_effective_date','row_expiry_date','is_current'
]]

print(f'dim_customer rows: {len(dim_customer)}')
dim_customer.head(3)

dim_customer rows: 599


,customer_id,first_name,last_name,email,address,district,city,country,postal_code,phone,active,create_date,row_effective_date,row_expiry_date,is_current
0,1,MARY,SMITH,MARY.SMITH@sakilacustomer.org,1913 Hanoi Way,Nagasaki,Sasebo,Japan,35200,28303384290,1,2006-02-14,2026-05-16,None,1
1,2,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,1121 Loja Avenue,California,San Bernardino,United States,17886,838635286649,1,2006-02-14,2026-05-16,None,1
2,3,LINDA,WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,692 Joliet Street,Attika,Athenai,Greece,83579,448477190408,1,2006-02-14,2026-05-16,None,1


### 3.3 — dim_film
Source: film + language (joined twice for language and original_language).

In [6]:
# select only needed columns from film
film_c = film[[
    'film_id','title','description','release_year',
    'language_id','original_language_id',
    'rental_duration','rental_rate','length',
    'replacement_cost','rating','special_features'
]].copy()

film_c['original_language_id'] = film_c['original_language_id'].astype('Int64')

lang_main = language[['language_id','name']].rename(columns={'name':'language'})
lang_orig = language[['language_id','name']].rename(columns={'name':'original_language'})
lang_orig['language_id'] = lang_orig['language_id'].astype('Int64')

# first merge: display language
dim_film = film_c.merge(lang_main, on='language_id', how='left')

# second merge: original language
dim_film = dim_film.merge(
    lang_orig,
    left_on='original_language_id',
    right_on='language_id',
    how='left',
    suffixes=('','_orig')
)

dim_film = dim_film[[
    'film_id','title','description','release_year',
    'language','original_language',
    'rental_duration','rental_rate','length',
    'replacement_cost','rating','special_features'
]].copy()

dim_film = dim_film.rename(columns={'length':'length_minutes'})

# fill missing values
dim_film['description']       = dim_film['description'].fillna('No description')
dim_film['original_language'] = dim_film['original_language'].fillna('Unknown')
dim_film['rating']            = dim_film['rating'].fillna('NR')
dim_film['special_features']  = dim_film['special_features'].fillna('None')
dim_film['length_minutes']    = dim_film['length_minutes'].fillna(0).astype(int)

dim_film = dim_film.drop_duplicates(subset='film_id')

print(f'dim_film rows: {len(dim_film)}')
dim_film.head(3)

dim_film rows: 1000


,film_id,title,description,release_year,language,original_language,rental_duration,rental_rate,length_minutes,replacement_cost,rating,special_features
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,English,Unknown,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes"
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,English,Unknown,3,4.99,48,12.99,G,"Trailers,Deleted Scenes"
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,English,Unknown,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes"


### 3.4 — dim_category
Source: category.

In [7]:
dim_category = category[['category_id','name']].copy()
dim_category = dim_category.drop_duplicates(subset='category_id')

print(f'dim_category rows: {len(dim_category)}')
dim_category

dim_category rows: 16


,category_id,name
0,1,Action
1,2,Animation
2,3,Children
3,4,Classics
4,5,Comedy
5,6,Documentary
6,7,Drama
7,8,Family
8,9,Foreign
9,10,Games


### 3.5 — dim_store
Source: store + staff (manager) + address + city + country (flattened).

In [8]:
# select only needed columns
store_c   = store  [['store_id','manager_staff_id','address_id']]
address_c = address[['address_id','address','district','postal_code','phone','city_id']]
city_c    = city   [['city_id','city','country_id']]
country_c = country[['country_id','country']]

# build manager name
manager = staff[['staff_id','first_name','last_name']].copy()
manager['manager_name'] = manager['first_name'] + ' ' + manager['last_name']
manager = manager.rename(columns={'staff_id':'manager_staff_id'})[['manager_staff_id','manager_name']]

dim_store = (
    store_c
    .merge(manager,   on='manager_staff_id')
    .merge(address_c, on='address_id')
    .merge(city_c,    on='city_id')
    .merge(country_c, on='country_id')
)

dim_store['postal_code'] = dim_store['postal_code'].fillna('00000')
dim_store = dim_store.drop_duplicates(subset='store_id')

# keep only columns that exist in the DW table
dim_store = dim_store[[
    'store_id','address','district','city',
    'country','postal_code','phone','manager_name'
]]

print(f'dim_store rows: {len(dim_store)}')
dim_store

dim_store rows: 2


,store_id,address,district,city,country,postal_code,phone,manager_name
0,1,47 MySakila Drive,Alberta,Lethbridge,Canada,,,Mike Hillyer
1,2,28 MySQL Boulevard,QLD,Woodridge,Australia,,,Jon Stephens


### 3.6 — dim_staff
Source: staff.

In [9]:
dim_staff = staff[['staff_id','first_name','last_name','email','username','store_id','active']].copy()
dim_staff['full_name'] = dim_staff['first_name'] + ' ' + dim_staff['last_name']
dim_staff['email']     = dim_staff['email'].fillna('no-email@unknown.com')
dim_staff['active']    = dim_staff['active'].astype(int)
dim_staff = dim_staff.drop_duplicates(subset='staff_id')

print(f'dim_staff rows: {len(dim_staff)}')
dim_staff

dim_staff rows: 2


,staff_id,first_name,last_name,email,username,store_id,active,full_name
0,1,Mike,Hillyer,Mike.Hillyer@sakilastaff.com,Mike,1,1,Mike Hillyer
1,2,Jon,Stephens,Jon.Stephens@sakilastaff.com,Jon,2,1,Jon Stephens


### 3.7 — dim_actor
Source: actor.

In [10]:
dim_actor = actor[['actor_id','first_name','last_name']].copy()
dim_actor['full_name'] = dim_actor['first_name'] + ' ' + dim_actor['last_name']
dim_actor = dim_actor.drop_duplicates(subset='actor_id')

print(f'dim_actor rows: {len(dim_actor)}')
dim_actor.head(3)

dim_actor rows: 200


,actor_id,first_name,last_name,full_name
0,1,PENELOPE,GUINESS,PENELOPE GUINESS
1,2,NICK,WAHLBERG,NICK WAHLBERG
2,3,ED,CHASE,ED CHASE


### 3.8 — fact_rental
Source: rental + inventory + film + film_category.



In [11]:
# select only needed columns before merging to avoid last_update conflicts
rental_c    = rental   [['rental_id','rental_date','return_date','customer_id','inventory_id','staff_id']]
inventory_c = inventory[['inventory_id','film_id','store_id']]
film_c      = film     [['film_id','rental_duration','rental_rate','replacement_cost']]


film_cat_c = (
    film_cat[['film_id','category_id']]
    .groupby('film_id', as_index=False)
    .first()
)

fact_rental = (
    rental_c
    .merge(inventory_c, on='inventory_id')
    .merge(film_c,      on='film_id')
    .merge(film_cat_c,  on='film_id')
)

# convert dates
fact_rental['rental_date'] = pd.to_datetime(fact_rental['rental_date'])
fact_rental['return_date'] = pd.to_datetime(fact_rental['return_date'])

# date keys as YYYYMMDD integers
fact_rental['rental_date_key'] = fact_rental['rental_date'].dt.strftime('%Y%m%d').astype(int)
fact_rental['return_date_key'] = fact_rental['return_date'].apply(
    lambda d: int(d.strftime('%Y%m%d')) if pd.notna(d) else None
)

# MySQL SMALLINT cannot accept Python float NaN — it raises a data type error
fact_rental['rental_duration_days'] = (
    (fact_rental['return_date'] - fact_rental['rental_date']).dt.days
).astype('Int16')

fact_rental['allowed_duration_days'] = fact_rental['rental_duration'].astype(int)


overdue_raw = np.where(
    fact_rental['return_date'].isna(),
    pd.NA,
    np.maximum(0, fact_rental['rental_duration_days'].astype('float') - fact_rental['rental_duration'])
)
fact_rental['days_overdue'] = pd.array(overdue_raw, dtype='Int16')

fact_rental['is_returned'] = fact_rental['return_date'].notna().astype(int)

fact_rental = fact_rental.drop_duplicates(subset='rental_id')

# keep only columns needed
fact_rental = fact_rental[[
    'rental_date_key','return_date_key',
    'customer_id','film_id','category_id','store_id','staff_id',
    'rental_id',
    'rental_duration_days','allowed_duration_days',
    'days_overdue','rental_rate','replacement_cost','is_returned'
]]

print(f'fact_rental rows: {len(fact_rental)}')
fact_rental.head(3)

fact_rental rows: 16044


,rental_date_key,return_date_key,customer_id,film_id,category_id,store_id,staff_id,rental_id,rental_duration_days,allowed_duration_days,days_overdue,rental_rate,replacement_cost,is_returned
0,20050524,20050526.0,130,80,8,1,1,1,1,7,0,2.99,21.99,1
1,20050524,20050528.0,459,333,12,2,1,2,3,7,0,2.99,16.99,1
2,20050524,20050601.0,408,373,3,2,1,3,7,7,0,2.99,14.99,1


### 3.9 — fact_payment
Source: payment + staff (for store_id).

> **Fix 5:** `payment_id` cast to `Int16` — DDL declares it `SMALLINT`; pandas default is `int64`.

In [12]:
# select only needed columns
payment_c = payment[['payment_id','customer_id','staff_id','rental_id','amount','payment_date']].copy()
staff_c   = staff  [['staff_id','store_id']]

fact_payment = payment_c.merge(staff_c, on='staff_id', how='left')

# date key
fact_payment['payment_date']     = pd.to_datetime(fact_payment['payment_date'])
fact_payment['payment_date_key'] = fact_payment['payment_date'].dt.strftime('%Y%m%d').astype(int)

# rename amount
fact_payment = fact_payment.rename(columns={'amount':'amount_paid'})


fact_payment['payment_id'] = fact_payment['payment_id'].astype('Int16')

# remove duplicates
fact_payment = fact_payment.drop_duplicates(subset='payment_id')

fact_payment = fact_payment[[
    'payment_date_key','customer_id','staff_id','store_id',
    'rental_id','payment_id','amount_paid'
]]

print(f'fact_payment rows: {len(fact_payment)}')
fact_payment.head(3)

fact_payment rows: 16044


,payment_date_key,customer_id,staff_id,store_id,rental_id,payment_id,amount_paid
0,20050525,1,1,1,76,1,2.99
1,20050528,1,1,1,573,2,0.99
2,20050615,1,1,1,1185,3,5.99


### 3.10 — fact_inventory_snapshot  
Source: inventory + rental.



In [13]:
# ── total copies per film × store (from inventory) ───────────
total_copies = (
    inventory[['inventory_id','film_id','store_id']]
    .groupby(['film_id','store_id'], as_index=False)
    .agg(total_copies=('inventory_id','count'))
)

# ── rentals with their rental_date and return_date ───────────
rent_inv = (
    rental[['rental_id','inventory_id','rental_date','return_date']]
    .merge(inventory[['inventory_id','film_id','store_id']], on='inventory_id')
)
rent_inv['rental_date'] = pd.to_datetime(rent_inv['rental_date']).dt.normalize()
rent_inv['return_date'] = pd.to_datetime(rent_inv['return_date']).dt.normalize()

# ── get every distinct rental date that appears in the OLTP ──
snapshot_dates = rent_inv['rental_date'].dropna().unique()

# ── for each snapshot date: count how many copies were out ───
# A copy is "rented out" on date D if:  rental_date <= D  AND  (return_date > D OR return_date is NaT)
rows = []
for snap_date in sorted(snapshot_dates):
    mask = (
        (rent_inv['rental_date'] <= snap_date) &
        (rent_inv['return_date'].isna() | (rent_inv['return_date'] > snap_date))
    )
    rented = (
        rent_inv[mask]
        .groupby(['film_id','store_id'], as_index=False)
        .agg(copies_rented_out=('inventory_id','count'))
    )
    rented['snapshot_date'] = snap_date
    rows.append(rented)

rented_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(
    columns=['film_id','store_id','copies_rented_out','snapshot_date']
)

# ── join total_copies and compute available ───────────────────
fact_inventory_snapshot = rented_df.merge(total_copies, on=['film_id','store_id'], how='left')
fact_inventory_snapshot['total_copies']     = fact_inventory_snapshot['total_copies'].fillna(0).astype(int)
fact_inventory_snapshot['copies_rented_out']= fact_inventory_snapshot['copies_rented_out'].fillna(0).astype(int)
fact_inventory_snapshot['copies_available'] = (
    fact_inventory_snapshot['total_copies'] - fact_inventory_snapshot['copies_rented_out']
).clip(lower=0)

# date key
fact_inventory_snapshot['snapshot_date_key'] = (
    pd.to_datetime(fact_inventory_snapshot['snapshot_date']).dt.strftime('%Y%m%d').astype(int)
)

fact_inventory_snapshot = fact_inventory_snapshot[[
    'snapshot_date_key','film_id','store_id',
    'total_copies','copies_rented_out','copies_available'
]]

print(f'fact_inventory_snapshot rows: {len(fact_inventory_snapshot)}')
fact_inventory_snapshot.head(5)

fact_inventory_snapshot rows: 31807


,snapshot_date_key,film_id,store_id,total_copies,copies_rented_out,copies_available
0,20050524,80,1,4,1,3
1,20050524,333,2,2,1,1
2,20050524,373,2,2,1,1
3,20050524,450,2,4,1,3
4,20050524,510,1,4,1,3


---
## 4 — LOAD
Clears all DW tables then loads dimensions → bridge → facts in correct FK order.

> **New:** `fact_inventory_snapshot` added to TRUNCATE list and loaded after dimensions.

In [14]:
# ═══════════════════════════════════════════════════════════
# 4 — LOAD  (clear → dimensions → bridge → facts)
# ═══════════════════════════════════════════════════════════

# ── CLEAR all tables (reverse FK order) ─────────────────────
with dw_engine.connect() as conn:
    conn.execute(text('SET FOREIGN_KEY_CHECKS = 0'))
    for tbl in ['fact_payment','fact_rental','fact_inventory_snapshot',
                'bridge_film_actor',
                'dim_actor','dim_staff','dim_store',
                'dim_category','dim_film','dim_customer','dim_date']:
        conn.execute(text(f'TRUNCATE TABLE {tbl}'))
    conn.execute(text('SET FOREIGN_KEY_CHECKS = 1'))
    conn.commit()
print('All DW tables cleared ✓')

# ── LOAD DIMENSIONS ─────────────────────────────────────────
dim_date.to_sql('dim_date',         con=dw_engine, if_exists='append', index=False)
print(f'dim_date         : {len(dim_date)} rows ✓')

dim_customer.to_sql('dim_customer',  con=dw_engine, if_exists='append', index=False)
print(f'dim_customer     : {len(dim_customer)} rows ✓')

dim_film.to_sql('dim_film',          con=dw_engine, if_exists='append', index=False)
print(f'dim_film         : {len(dim_film)} rows ✓')

dim_category.to_sql('dim_category',  con=dw_engine, if_exists='append', index=False)
print(f'dim_category     : {len(dim_category)} rows ✓')

dim_store.to_sql('dim_store',        con=dw_engine, if_exists='append', index=False)
print(f'dim_store        : {len(dim_store)} rows ✓')

dim_staff.to_sql('dim_staff',        con=dw_engine, if_exists='append', index=False)
print(f'dim_staff        : {len(dim_staff)} rows ✓')

dim_actor.to_sql('dim_actor',        con=dw_engine, if_exists='append', index=False)
print(f'dim_actor        : {len(dim_actor)} rows ✓')

# ── LOAD BRIDGE ─────────────────────────────────────────────
dw_film_keys  = pd.read_sql('SELECT film_key,  film_id  FROM dim_film',  dw_engine)
dw_actor_keys = pd.read_sql('SELECT actor_key, actor_id FROM dim_actor', dw_engine)

bridge = (
    film_act[['film_id','actor_id']]
    .merge(dw_film_keys,  on='film_id')
    .merge(dw_actor_keys, on='actor_id')
    [['film_key','actor_key']]
    .drop_duplicates()
)
bridge.to_sql('bridge_film_actor', con=dw_engine, if_exists='append', index=False)
print(f'bridge_film_actor: {len(bridge)} rows ✓')

# ── LOAD fact_rental ────────────────────────────────────────
dw_cust = pd.read_sql('SELECT customer_key, customer_id FROM dim_customer WHERE is_current=1', dw_engine)
dw_flm  = pd.read_sql('SELECT film_key,     film_id     FROM dim_film',      dw_engine)
dw_cat  = pd.read_sql('SELECT category_key, category_id FROM dim_category',  dw_engine)
dw_str  = pd.read_sql('SELECT store_key,    store_id    FROM dim_store',      dw_engine)
dw_stf  = pd.read_sql('SELECT staff_key,    staff_id    FROM dim_staff',      dw_engine)

fr = (
    fact_rental
    .merge(dw_cust, on='customer_id')
    .merge(dw_flm,  on='film_id')
    .merge(dw_cat,  on='category_id')
    .merge(dw_str,  on='store_id')
    .merge(dw_stf,  on='staff_id')
)[[
    'rental_date_key','return_date_key',
    'customer_key','film_key','category_key','store_key','staff_key',
    'rental_id',
    'rental_duration_days','allowed_duration_days',
    'days_overdue','rental_rate','replacement_cost','is_returned'
]]

# data quality assertion: no rows should be lost during surrogate key join
assert len(fr) == len(fact_rental), (
    f'fact_rental row count mismatch: expected {len(fact_rental)}, got {len(fr)}'
)

fr.to_sql('fact_rental', con=dw_engine, if_exists='append', index=False)
print(f'fact_rental      : {len(fr)} rows ✓')

# ── LOAD fact_payment ───────────────────────────────────────
dw_cust2  = pd.read_sql('SELECT customer_key, customer_id FROM dim_customer WHERE is_current=1', dw_engine)
dw_stf2   = pd.read_sql('SELECT staff_key,    staff_id    FROM dim_staff',   dw_engine)
dw_str2   = pd.read_sql('SELECT store_key,    store_id    FROM dim_store',   dw_engine)
dw_rental = pd.read_sql('SELECT rental_key,   rental_id   FROM fact_rental', dw_engine)

fp = (
    fact_payment
    .merge(dw_cust2,  on='customer_id')
    .merge(dw_stf2,   on='staff_id')
    .merge(dw_str2,   on='store_id')
    .merge(dw_rental, on='rental_id', how='left')
)[[
    'payment_date_key',
    'customer_key','staff_key','store_key',
    'rental_key','payment_id','amount_paid'
]]

# data quality assertion
assert len(fp) == len(fact_payment), (
    f'fact_payment row count mismatch: expected {len(fact_payment)}, got {len(fp)}'
)

fp.to_sql('fact_payment', con=dw_engine, if_exists='append', index=False)
print(f'fact_payment     : {len(fp)} rows ✓')

# ── LOAD fact_inventory_snapshot (NEW) ──────────────────────
dw_flm2 = pd.read_sql('SELECT film_key,  film_id  FROM dim_film',  dw_engine)
dw_str3 = pd.read_sql('SELECT store_key, store_id FROM dim_store', dw_engine)

fi = (
    fact_inventory_snapshot
    .merge(dw_flm2, on='film_id')
    .merge(dw_str3, on='store_id')
)[[
    'snapshot_date_key','film_key','store_key',
    'total_copies','copies_rented_out','copies_available'
]]

fi.to_sql('fact_inventory_snapshot', con=dw_engine, if_exists='append', index=False)
print(f'fact_inv_snapshot: {len(fi)} rows ✓')

All DW tables cleared ✓
dim_date         : 2191 rows ✓
dim_customer     : 599 rows ✓
dim_film         : 1000 rows ✓
dim_category     : 16 rows ✓
dim_store        : 2 rows ✓
dim_staff        : 2 rows ✓
dim_actor        : 200 rows ✓
bridge_film_actor: 5462 rows ✓
fact_rental      : 16044 rows ✓
fact_payment     : 16044 rows ✓
fact_inv_snapshot: 31807 rows ✓


---
## 5 — VERIFY
Row counts for every table in sakila_dw.

> **New:** `fact_inventory_snapshot` added to the verification list.

In [15]:
# ═══════════════════════════════════════════════════════════
# 5 — VERIFY row counts
# ═══════════════════════════════════════════════════════════

tables = [
    'dim_date','dim_customer','dim_film','dim_category',
    'dim_store','dim_staff','dim_actor',
    'bridge_film_actor',
    'fact_rental','fact_payment','fact_inventory_snapshot'
]

print(f'{"Table":<30} {"Rows":>8}')
print('-' * 40)

for tbl in tables:
    n = pd.read_sql(
        f'SELECT COUNT(*) AS cnt FROM {tbl}',
        dw_engine
    ).iloc[0]['cnt']
    print(f'{tbl:<30} {int(n):>8}')

print()
print('All tables verified ✓')

Table                              Rows
----------------------------------------
dim_date                           2191
dim_customer                        599
dim_film                           1000
dim_category                         16
dim_store                             2
dim_staff                             2
dim_actor                           200
bridge_film_actor                  5462
fact_rental                       16044
fact_payment                      16044
fact_inventory_snapshot           31807

All tables verified ✓
